In [ ]:
# =====================================================
# CELL-1 : SETUP + DATASET + DUAL SSL AUGMENTATION
# =====================================================

import os
import random
import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# -----------------------------------------------------
# DEVICE
# -----------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# -----------------------------------------------------
# PATHS
# -----------------------------------------------------

TRAIN_PATH = "/kaggle/input/datasets/sabbir4724/training-data"
TEST_PATH  = "/kaggle/input/datasets/sabbir4724/testing-data"

# -----------------------------------------------------
# REPRODUCIBILITY
# -----------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# -----------------------------------------------------
# SSL AUGMENTATION VIEW-1
# -----------------------------------------------------

ssl_transform_1 = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(
        224,
        scale=(0.8,1.0)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

# -----------------------------------------------------
# SSL AUGMENTATION VIEW-2
# -----------------------------------------------------

ssl_transform_2 = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.RandomResizedCrop(
        224,
        scale=(0.7,1.0)
    ),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

# -----------------------------------------------------
# TEST TRANSFORM
# -----------------------------------------------------

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

# -----------------------------------------------------
# SSL DATASET
# RETURNS:
# view1, view2, label
# -----------------------------------------------------

class SSLDataset(Dataset):

    def __init__(self, root):

        self.root = root

        self.classes = sorted([
            d for d in os.listdir(root)
            if os.path.isdir(os.path.join(root,d))
        ])

        self.class_to_idx = {
            c:i for i,c in enumerate(self.classes)
        }

        self.images = []
        self.labels = []

        for cls in self.classes:

            folder = os.path.join(root, cls)

            for img in os.listdir(folder):

                img_path = os.path.join(folder, img)

                self.images.append(img_path)
                self.labels.append(
                    self.class_to_idx[cls]
                )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img_path = self.images[idx]

        img = Image.open(
            img_path
        ).convert("RGB")

        view1 = ssl_transform_1(img)
        view2 = ssl_transform_2(img)

        label = self.labels[idx]

        return view1, view2, label


# -----------------------------------------------------
# TEST DATASET
# -----------------------------------------------------

class TestDataset(Dataset):

    def __init__(self, root):

        self.root = root

        self.classes = sorted([
            d for d in os.listdir(root)
            if os.path.isdir(os.path.join(root,d))
        ])

        self.class_to_idx = {
            c:i for i,c in enumerate(self.classes)
        }

        self.images = []
        self.labels = []

        for cls in self.classes:

            folder = os.path.join(root, cls)

            for img in os.listdir(folder):

                img_path = os.path.join(folder, img)

                self.images.append(img_path)
                self.labels.append(
                    self.class_to_idx[cls]
                )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img = Image.open(
            self.images[idx]
        ).convert("RGB")

        img = test_transform(img)

        label = self.labels[idx]

        return img, label


# -----------------------------------------------------
# LOAD DATASETS
# -----------------------------------------------------

train_dataset = SSLDataset(TRAIN_PATH)

test_dataset = TestDataset(TEST_PATH)

# -----------------------------------------------------
# DATALOADERS
# -----------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# -----------------------------------------------------
# INFO
# -----------------------------------------------------

print("\nClasses:")
print(train_dataset.classes)

print("\nClass Mapping:")
print(train_dataset.class_to_idx)

print("\nTrain Images:", len(train_dataset))
print("Test Images :", len(test_dataset))

# -----------------------------------------------------
# SANITY CHECK
# -----------------------------------------------------

v1, v2, y = train_dataset[0]

print("\nSSL View-1 Shape :", v1.shape)
print("SSL View-2 Shape :", v2.shape)
print("Label            :", y)


In [ ]:

# CELL-2 : MobileNetV3 + ECA + Projection Head

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

# -----------------------------------------------------
# ECA ATTENTION
# -----------------------------------------------------

class ECALayer(nn.Module):

    def __init__(self, channels, k_size=3):
        super().__init__()

        self.avg_pool = nn.AdaptiveAvgPool2d(1)

        self.conv = nn.Conv1d(
            1,
            1,
            kernel_size=k_size,
            padding=(k_size - 1) // 2,
            bias=False
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        y = self.avg_pool(x)

        y = y.squeeze(-1).transpose(-1, -2)

        y = self.conv(y)

        y = self.sigmoid(y)

        y = y.transpose(-1, -2).unsqueeze(-1)

        return x * y.expand_as(x)


# -----------------------------------------------------
# ENCODER
# -----------------------------------------------------

class MobileNetV3_ECA(nn.Module):

    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            "mobilenetv3_large_100",
            pretrained=True,
            num_classes=0
        )

        self.eca = ECALayer(
            channels=960,
            k_size=3
        )

    def forward(self, x):

        # Feature Map
        feat_map = self.backbone.forward_features(x)

        # ECA Attention
        feat_map = self.eca(feat_map)

        # Global Pooling
        feat = F.adaptive_avg_pool2d(
            feat_map,
            1
        )

        feat = feat.view(
            feat.size(0),
            -1
        )

        return feat


# -----------------------------------------------------
# PROJECTION HEAD
# -----------------------------------------------------

class ProjectionHead(nn.Module):

    def __init__(
        self,
        in_dim=960,
        hidden_dim=512,
        out_dim=128
    ):
        super().__init__()

        self.projector = nn.Sequential(

            nn.Linear(
                in_dim,
                hidden_dim
            ),

            nn.BatchNorm1d(
                hidden_dim
            ),

            nn.ReLU(inplace=True),

            nn.Linear(
                hidden_dim,
                out_dim
            )
        )

    def forward(self, x):
        return self.projector(x)


# -----------------------------------------------------
# SSL MODEL
# -----------------------------------------------------

class SSLModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = MobileNetV3_ECA()

        self.projector = ProjectionHead()

    def forward(self, x):

        feat = self.encoder(x)

        proj = self.projector(feat)

        return feat, proj


# -----------------------------------------------------
# BUILD MODEL
# -----------------------------------------------------

ssl_model = SSLModel().to(device)

# -----------------------------------------------------
# PARAMETER COUNT
# -----------------------------------------------------

total_params = sum(
    p.numel()
    for p in ssl_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in ssl_model.parameters()
    if p.requires_grad
)

print("\nModel Summary")
print("="*50)

print(
    f"Total Params     : {total_params:,}"
)

print(
    f"Trainable Params : {trainable_params:,}"
)

# -----------------------------------------------------
# SANITY CHECK
# -----------------------------------------------------

sample_v1, sample_v2, _ = next(
    iter(train_loader)
)

sample_v1 = sample_v1.to(device)

with torch.no_grad():

    features, projections = ssl_model(
        sample_v1
    )

print("\nFeature Shape :", features.shape)
print("Projection Shape :", projections.shape)



In [ ]:
# =====================================================
# CELL-3 : SSL PRETRAINING (UPDATED)
# NT-Xent + Loss Curve + Best Model Save
# =====================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm

# -----------------------------------------------------
# NT-XENT LOSS
# -----------------------------------------------------

class NTXentLoss(nn.Module):

    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature

    def forward(self, z1, z2):

        batch_size = z1.size(0)

        z1 = F.normalize(z1, dim=1)
        z2 = F.normalize(z2, dim=1)

        representations = torch.cat(
            [z1, z2],
            dim=0
        )

        similarity_matrix = torch.matmul(
            representations,
            representations.T
        )

        similarity_matrix = (
            similarity_matrix /
            self.temperature
        )

        mask = torch.eye(
            2 * batch_size,
            dtype=torch.bool
        ).to(z1.device)

        similarity_matrix = similarity_matrix.masked_fill(
            mask,
            -1e9
        )

        positives = torch.cat([
            torch.diag(similarity_matrix, batch_size),
            torch.diag(similarity_matrix, -batch_size)
        ])

        numerator = torch.exp(positives)

        denominator = torch.sum(
            torch.exp(similarity_matrix),
            dim=1
        )

        loss = -torch.log(
            numerator / denominator
        )

        return loss.mean()


# -----------------------------------------------------
# OPTIMIZER
# -----------------------------------------------------

optimizer = torch.optim.AdamW(
    ssl_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

criterion = NTXentLoss(
    temperature=0.5
)

# -----------------------------------------------------
# CONFIG
# -----------------------------------------------------

EPOCHS = 20

best_loss = float("inf")

save_path = "ssl_encoder.pth"

ssl_losses = []

# -----------------------------------------------------
# TRAINING
# -----------------------------------------------------

for epoch in range(EPOCHS):

    ssl_model.train()

    running_loss = 0.0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for view1, view2, _ in pbar:

        view1 = view1.to(device)
        view2 = view2.to(device)

        optimizer.zero_grad()

        _, z1 = ssl_model(view1)
        _, z2 = ssl_model(view2)

        loss = criterion(z1, z2)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        pbar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    epoch_loss = (
        running_loss /
        len(train_loader)
    )

    ssl_losses.append(epoch_loss)

    print(
        f"\nEpoch [{epoch+1}/{EPOCHS}] "
        f"SSL Loss: {epoch_loss:.4f}"
    )

    if epoch_loss < best_loss:

        best_loss = epoch_loss

        torch.save(
            ssl_model.state_dict(),
            save_path
        )

        print(
            f"✅ Best model saved "
            f"(Loss={epoch_loss:.4f})"
        )

# -----------------------------------------------------
# LOAD BEST MODEL
# -----------------------------------------------------

ssl_model.load_state_dict(
    torch.load(
        save_path,
        map_location=device
    )
)

print("\n==================================================")
print("SSL PRETRAINING FINISHED")
print("==================================================")
print(f"Best SSL Loss : {best_loss:.4f}")
print(f"Saved Model   : {save_path}")

# -----------------------------------------------------
# LOSS CURVE
# -----------------------------------------------------

plt.figure(figsize=(8,5))

plt.plot(
    range(1, len(ssl_losses)+1),
    ssl_losses,
    marker='o',
    linewidth=2
)

plt.xlabel("Epoch")
plt.ylabel("NT-Xent Loss")
plt.title("SSL Pretraining Loss Curve")

plt.grid(True)

plt.savefig(
    "ssl_loss_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSaved Figure : ssl_loss_curve.png")

# -----------------------------------------------------
# FINAL LOSS TABLE
# -----------------------------------------------------

for i, loss in enumerate(ssl_losses):
    print(
        f"Epoch {i+1:02d} : {loss:.4f}"
    )



In [ ]:

# CELL-3.5 : SUPERVISED FINE-TUNING


import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from PIL import Image
from torch.utils.data import DataLoader

# =====================================================
# OUTPUT FOLDER
# =====================================================

OUTPUT_DIR = "/kaggle/working/output"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print(f"Output Folder: {OUTPUT_DIR}")

# =====================================================
# DATASET
# =====================================================

class ClassificationDataset(torch.utils.data.Dataset):

    def __init__(self, root):

        self.classes = sorted([
            d for d in os.listdir(root)
            if os.path.isdir(os.path.join(root, d))
        ])

        self.images = []
        self.labels = []

        for idx, cls in enumerate(self.classes):

            folder = os.path.join(root, cls)

            for img in os.listdir(folder):

                self.images.append(
                    os.path.join(folder, img)
                )

                self.labels.append(idx)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img = Image.open(
            self.images[idx]
        ).convert("RGB")

        img = test_transform(img)

        label = self.labels[idx]

        return img, label

# =====================================================
# TRAIN DATASET
# =====================================================

train_cls_ds = ClassificationDataset(
    TRAIN_PATH
)

train_cls_loader = DataLoader(
    train_cls_ds,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print(
    f"Training Images: {len(train_cls_ds)}"
)

# =====================================================
# MODEL
# =====================================================

class FineTuneModel(nn.Module):

    def __init__(self, ssl_model):

        super().__init__()

        self.encoder = ssl_model.encoder

        self.classifier = nn.Linear(
            960,
            4
        )

    def forward(self, x):

        feat = self.encoder(x)

        out = self.classifier(feat)

        return out

# =====================================================
# LOAD SSL WEIGHTS
# =====================================================

ssl_model.load_state_dict(

    torch.load(
        "ssl_encoder.pth",
        map_location=device
    )

)

print("SSL Weights Loaded")

# =====================================================
# BUILD MODEL
# =====================================================

finetune_model = FineTuneModel(
    ssl_model
).to(device)

# =====================================================
# PARAM COUNT
# =====================================================

total_params = sum(
    p.numel()
    for p in finetune_model.parameters()
)

print(
    f"Total Parameters: {total_params:,}"
)

# =====================================================
# LOSS + OPTIMIZER
# =====================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(

    finetune_model.parameters(),

    lr=1e-4,

    weight_decay=1e-4
)

# =====================================================
# TRAIN CONFIG
# =====================================================

EPOCHS = 30

best_acc = 0

train_losses = []
train_accs = []

# =====================================================
# TRAINING
# =====================================================

for epoch in range(EPOCHS):

    finetune_model.train()

    running_loss = 0

    correct = 0

    total = 0

    pbar = tqdm(
        train_cls_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for imgs, labels in pbar:

        imgs = imgs.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = finetune_model(imgs)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        preds = outputs.argmax(1)

        correct += (
            preds == labels
        ).sum().item()

        total += labels.size(0)

        pbar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    epoch_loss = (
        running_loss /
        len(train_cls_loader)
    )

    epoch_acc = (
        correct /
        total
    )

    train_losses.append(
        epoch_loss
    )

    train_accs.append(
        epoch_acc
    )

    print(
        f"Epoch {epoch+1:02d}"
        f" | Loss={epoch_loss:.4f}"
        f" | Acc={epoch_acc:.4f}"
    )

    # -----------------------------------------
    # SAVE BEST MODEL
    # -----------------------------------------

    if epoch_acc > best_acc:

        best_acc = epoch_acc

        torch.save(

            finetune_model.encoder.state_dict(),

            os.path.join(
                OUTPUT_DIR,
                "finetuned_encoder.pth"
            )
        )

        print(
            "✅ Best Encoder Saved"
        )

# =====================================================
# TRAIN HISTORY TABLE
# =====================================================

history_df = pd.DataFrame({

    "Epoch":
    list(range(1, EPOCHS+1)),

    "Train_Loss":
    train_losses,

    "Train_Accuracy":
    train_accs

})

history_csv = os.path.join(
    OUTPUT_DIR,
    "finetuning_history.csv"
)

history_df.to_csv(
    history_csv,
    index=False
)

# =====================================================
# LOSS CURVE
# =====================================================

plt.figure(figsize=(8,5))

plt.plot(
    train_losses,
    marker="o"
)

plt.title(
    "Fine-Tuning Loss"
)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.grid(True)

loss_fig = os.path.join(
    OUTPUT_DIR,
    "finetuning_loss_curve.png"
)

plt.savefig(
    loss_fig,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# =====================================================
# ACC CURVE
# =====================================================

plt.figure(figsize=(8,5))

plt.plot(
    train_accs,
    marker="o"
)

plt.title(
    "Fine-Tuning Accuracy"
)

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.grid(True)

acc_fig = os.path.join(
    OUTPUT_DIR,
    "finetuning_accuracy_curve.png"
)

plt.savefig(
    acc_fig,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# =====================================================
# FINAL SUMMARY TABLE
# =====================================================

summary_df = pd.DataFrame({

    "Metric":[
        "Training Images",
        "Epochs",
        "Best Accuracy (%)",
        "Total Parameters"
    ],

    "Value":[
        len(train_cls_ds),
        EPOCHS,
        round(best_acc*100,4),
        f"{total_params:,}"
    ]

})

summary_csv = os.path.join(
    OUTPUT_DIR,
    "finetuning_summary.csv"
)

summary_df.to_csv(
    summary_csv,
    index=False
)

# =====================================================
# PRINT RESULTS
# =====================================================

print("\n")
print("="*60)
print("FINE-TUNING COMPLETE")
print("="*60)

print(
    f"Best Train Accuracy : {best_acc*100:.4f}%"
)

print(
    f"Total Parameters    : {total_params:,}"
)

print("\nSaved Files:")

print(
    "✓ finetuned_encoder.pth"
)

print(
    "✓ finetuning_history.csv"
)

print(
    "✓ finetuning_summary.csv"
)

print(
    "✓ finetuning_loss_curve.png"
)

print(
    "✓ finetuning_accuracy_curve.png"
)

print("\n")
print("="*60)
print("SUMMARY TABLE")
print("="*60)

display(summary_df)

print("\nOutput Directory:")
print(OUTPUT_DIR)

In [ ]:
# CELL-4 : PROTOTYPICAL NETWORK (FINAL VERSION)
# Fine-Tuned Encoder + 5-shot + External Test
# Metrics + Confusion Matrix + ROC + t-SNE


import os
import random
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from PIL import Image
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)

from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

# -----------------------------------------------------
# OUTPUT FOLDER (must match CELL-3.5)
# -----------------------------------------------------

OUTPUT_DIR = "/kaggle/working/output"

# -----------------------------------------------------
# DEVICE
# -----------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -----------------------------------------------------
# SANITY CHECK — confirm the file actually exists
# -----------------------------------------------------

encoder_path = os.path.join(OUTPUT_DIR, "finetuned_encoder.pth")

print("Looking for:", encoder_path)
print("File exists:", os.path.exists(encoder_path))

if not os.path.exists(encoder_path):
    print("Files currently in OUTPUT_DIR:")
    if os.path.exists(OUTPUT_DIR):
        print(os.listdir(OUTPUT_DIR))
    else:
        print("OUTPUT_DIR itself does not exist — re-run CELL-3.5 first.")
    raise FileNotFoundError(
        f"{encoder_path} not found. Re-run CELL-3.5 (fine-tuning) completely before this cell."
    )

# -----------------------------------------------------
# LOAD FINE-TUNED ENCODER
# -----------------------------------------------------

ssl_model.encoder.load_state_dict(
    torch.load(
        encoder_path,
        map_location=device
    )
)

ssl_model = ssl_model.to(device)
ssl_model.eval()

print("Fine-Tuned Encoder Loaded")

# -----------------------------------------------------
# DATASET
# -----------------------------------------------------

class FeatureDataset(Dataset):
    def __init__(self, root):
        self.classes = sorted([
            d for d in os.listdir(root)
            if os.path.isdir(os.path.join(root, d))
        ])

        self.images = []
        self.labels = []

        for idx, cls in enumerate(self.classes):
            folder = os.path.join(root, cls)

            for img in os.listdir(folder):
                self.images.append(os.path.join(folder, img))
                self.labels.append(idx)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        img = test_transform(img)
        return img, self.labels[idx]


train_ds = FeatureDataset(TRAIN_PATH)
test_ds  = FeatureDataset(TEST_PATH)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False)

# -----------------------------------------------------
# FEATURE EXTRACTION
# -----------------------------------------------------

def extract_features(loader):
    feats, labels = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            f = ssl_model.encoder(x)

            feats.append(f.cpu())
            labels.append(y)

    return torch.cat(feats), torch.cat(labels)


train_features, train_labels = extract_features(train_loader)
test_features, test_labels   = extract_features(test_loader)

print("Train:", train_features.shape)
print("Test :", test_features.shape)

# -----------------------------------------------------
# 5-SHOT SUPPORT SET
# -----------------------------------------------------

N_SHOT = 5

support_feats = []
support_lbls = []

classes = torch.unique(train_labels)

for cls in classes:
    idx = torch.where(train_labels == cls)[0]

    idx = idx[torch.randperm(len(idx))[:N_SHOT]]

    support_feats.append(train_features[idx])
    support_lbls.append(train_labels[idx])

support_feats = torch.cat(support_feats)
support_lbls = torch.cat(support_lbls)

print("Support Samples:", len(support_lbls))

# -----------------------------------------------------
# PROTOTYPES
# -----------------------------------------------------

prototypes = []

for cls in classes:
    proto = support_feats[support_lbls == cls].mean(0)
    prototypes.append(proto)

prototypes = torch.stack(prototypes)

print("Prototype Shape:", prototypes.shape)

# -----------------------------------------------------
# PREDICTION (PROTO-NET)
# -----------------------------------------------------

test_features_norm = F.normalize(test_features, dim=1)
prototypes_norm = F.normalize(prototypes, dim=1)

distances = torch.cdist(test_features_norm, prototypes_norm)
preds = torch.argmin(distances, dim=1)

# -----------------------------------------------------
# METRICS
# -----------------------------------------------------

y_true = test_labels.numpy()
y_pred = preds.numpy()

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average="weighted")
rec  = recall_score(y_true, y_pred, average="weighted")
f1   = f1_score(y_true, y_pred, average="weighted")

print("\n==============================")
print("PROTO-NET RESULTS")
print("==============================")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1 Score  : {f1:.4f}")

# -----------------------------------------------------
# CLASSIFICATION REPORT
# -----------------------------------------------------

print("\nClassification Report\n")
print(classification_report(
    y_true,
    y_pred,
    target_names=train_ds.classes,
    digits=4
))

# -----------------------------------------------------
# CONFUSION MATRIX
# -----------------------------------------------------

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.colorbar()

plt.xticks(range(len(train_ds.classes)), train_ds.classes, rotation=45)
plt.yticks(range(len(train_ds.classes)), train_ds.classes)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i,j], ha="center", va="center")

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()

cm_fig = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
plt.savefig(cm_fig, dpi=600, bbox_inches="tight")
plt.show()

# -----------------------------------------------------
# ROC CURVE
# -----------------------------------------------------

scores = torch.softmax(-distances, dim=1).numpy()

y_true_bin = label_binarize(y_true, classes=[0,1,2,3])

plt.figure(figsize=(8,6))

for i in range(4):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], scores[:, i])
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f"{train_ds.classes[i]} (AUC={roc_auc:.3f})")

plt.plot([0,1],[0,1],'k--')
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid()

roc_fig = os.path.join(OUTPUT_DIR, "roc_curve.png")
plt.savefig(roc_fig, dpi=600, bbox_inches="tight")
plt.show()

# -----------------------------------------------------
# t-SNE VISUALIZATION
# -----------------------------------------------------

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
feat_2d = tsne.fit_transform(test_features.numpy())

plt.figure(figsize=(8,6))

for cls in np.unique(y_true):
    idx = y_true == cls
    plt.scatter(feat_2d[idx,0], feat_2d[idx,1], label=train_ds.classes[cls], alpha=0.6)

plt.title("t-SNE Feature Space (SSL + Fine-Tuned)")
plt.legend()

tsne_fig = os.path.join(OUTPUT_DIR, "tsne_plot.png")
plt.savefig(tsne_fig, dpi=600, bbox_inches="tight")
plt.show()

# -----------------------------------------------------
# SAVE RESULTS SUMMARY
# -----------------------------------------------------

import pandas as pd

results_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],
    "Value (%)": [round(acc*100,4), round(prec*100,4), round(rec*100,4), round(f1*100,4)]
})

results_csv = os.path.join(OUTPUT_DIR, "protonet_results.csv")
results_df.to_csv(results_csv, index=False)

print("\nSaved:", results_csv)
display(results_df)

In [ ]:
# =====================================================
# CELL-5 : EXPLAINABLE AI — GRAD-CAM + GRAD-CAM++
# For MobileNetV3_ECA Encoder + Prototypical Network
# =====================================================

import os
import cv2
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image

# -----------------------------------------------------
# OUTPUT FOLDER FOR XAI
# -----------------------------------------------------

XAI_DIR = os.path.join(OUTPUT_DIR, "gradcam")
os.makedirs(XAI_DIR, exist_ok=True)

# -----------------------------------------------------
# TARGET LAYER
# Last feature map after ECA attention (960 channels)
# -----------------------------------------------------

target_layer = ssl_model.encoder.eca

# -----------------------------------------------------
# GRAD-CAM BASE CLASS
# -----------------------------------------------------

class GradCAMBase:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None

        self.h1 = target_layer.register_forward_hook(self._save_activation)
        self.h2 = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]

    def remove_hooks(self):
        self.h1.remove()
        self.h2.remove()

    def _postprocess(self, cam, img_size):
        cam = F.relu(cam)
        cam = F.interpolate(
            cam,
            size=img_size,
            mode="bilinear",
            align_corners=False
        )
        cam = cam.squeeze().detach().cpu().numpy()
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam


# -----------------------------------------------------
# GRAD-CAM
# -----------------------------------------------------

class GradCAM(GradCAMBase):
    def generate(self, score, img_size):
        self.model.zero_grad()
        score.backward(retain_graph=True)

        gradients = self.gradients          # [B,C,H,W]
        activations = self.activations      # [B,C,H,W]

        weights = gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * activations).sum(dim=1, keepdim=True)

        return self._postprocess(cam, img_size)


# -----------------------------------------------------
# GRAD-CAM++
# -----------------------------------------------------

class GradCAMPlusPlus(GradCAMBase):
    def generate(self, score, img_size):
        self.model.zero_grad()
        score.backward(retain_graph=True)

        gradients = self.gradients
        activations = self.activations

        grad2 = gradients ** 2
        grad3 = gradients ** 3

        sum_act = activations.sum(dim=(2, 3), keepdim=True)

        eps = 1e-8
        alpha_denom = 2 * grad2 + sum_act * grad3
        alpha_denom = torch.where(
            alpha_denom != 0,
            alpha_denom,
            torch.ones_like(alpha_denom) * eps
        )
        alpha = grad2 / (alpha_denom + eps)

        weights = (alpha * F.relu(gradients)).sum(dim=(2, 3), keepdim=True)
        cam = (weights * activations).sum(dim=1, keepdim=True)

        return self._postprocess(cam, img_size)


# -----------------------------------------------------
# PROTONET CLASS SCORE
# score = -distance(feat, prototype_c)  -> higher = closer match
# (same logic as cell-4's softmax(-distances))
# -----------------------------------------------------

def protonet_score(img_tensor, target_class_idx, prototypes_norm):
    feat = ssl_model.encoder(img_tensor)          # [1, 960], grad-enabled
    feat_norm = F.normalize(feat, dim=1)
    dist = torch.cdist(feat_norm, prototypes_norm)  # [1, num_classes]
    score = -dist[0, target_class_idx]
    return score


# -----------------------------------------------------
# OVERLAY HELPER
# -----------------------------------------------------

def overlay_cam(orig_img, cam, alpha=0.45):
    heatmap = cv2.applyColorMap(
        np.uint8(255 * cam), cv2.COLORMAP_JET
    )
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    heatmap = heatmap.astype(np.float32) / 255.0

    orig = np.array(orig_img).astype(np.float32) / 255.0
    overlay = alpha * heatmap + (1 - alpha) * orig
    overlay = np.clip(overlay, 0, 1)
    return overlay


# -----------------------------------------------------
# RUN GRAD-CAM + GRAD-CAM++ ON SAMPLE TEST IMAGES
# (1 correctly-classified image per class)
# -----------------------------------------------------

prototypes_norm = F.normalize(prototypes.to(device), dim=1)
class_names = train_ds.classes

sample_indices = {}
for i in range(len(test_ds)):
    lbl = test_ds.labels[i]
    if lbl not in sample_indices and y_pred[i] == lbl:
        sample_indices[lbl] = i
    if len(sample_indices) == len(class_names):
        break

fig, axes = plt.subplots(
    len(sample_indices), 3,
    figsize=(12, 4 * len(sample_indices))
)

if len(sample_indices) == 1:
    axes = axes.reshape(1, -1)

for row, (cls_idx, idx) in enumerate(sorted(sample_indices.items())):

    img_path = test_ds.images[idx]
    orig_img = Image.open(img_path).convert("RGB").resize((224, 224))

    img_tensor = test_transform(orig_img).unsqueeze(0).to(device)
    img_tensor.requires_grad_(True)

    # ---------- Grad-CAM ----------
    ssl_model.zero_grad()
    cam_extractor = GradCAM(ssl_model, target_layer)
    score = protonet_score(img_tensor, cls_idx, prototypes_norm)
    cam_map = cam_extractor.generate(score, img_size=(224, 224))
    cam_extractor.remove_hooks()

    # ---------- Grad-CAM++ ----------
    ssl_model.zero_grad()
    img_tensor2 = test_transform(orig_img).unsqueeze(0).to(device)
    img_tensor2.requires_grad_(True)
    campp_extractor = GradCAMPlusPlus(ssl_model, target_layer)
    score2 = protonet_score(img_tensor2, cls_idx, prototypes_norm)
    campp_map = campp_extractor.generate(score2, img_size=(224, 224))
    campp_extractor.remove_hooks()

    overlay_gc = overlay_cam(orig_img, cam_map)
    overlay_gcpp = overlay_cam(orig_img, campp_map)

    axes[row, 0].imshow(orig_img)
    axes[row, 0].set_title(f"{class_names[cls_idx]} - Original")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(overlay_gc)
    axes[row, 1].set_title("Grad-CAM")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(overlay_gcpp)
    axes[row, 2].set_title("Grad-CAM++")
    axes[row, 2].axis("off")

plt.tight_layout()

xai_fig_path = os.path.join(XAI_DIR, "gradcam_gradcampp_comparison.png")
plt.savefig(xai_fig_path, dpi=600, bbox_inches="tight")
plt.show()

print(f"✅ Saved: {xai_fig_path}")
print(f"Classes visualized: {[class_names[c] for c in sorted(sample_indices.keys())]}")

In [ ]:
# =====================================================
# CELL-6 : QUANTITATIVE XAI EVALUATION
# Insertion AUC (↑) + Deletion AUC (↓)
# =====================================================

import numpy as np
import cv2
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os

# -----------------------------------------------------
# CONFIG
# -----------------------------------------------------

N_STEPS   = 50          # number of insertion/deletion steps
BLUR_KSIZE = 31         # gaussian blur kernel for "missing" baseline
IMG_SIZE  = 224

MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

METRIC_DIR = os.path.join(OUTPUT_DIR, "gradcam")
os.makedirs(METRIC_DIR, exist_ok=True)

# -----------------------------------------------------
# HELPERS
# -----------------------------------------------------

def normalize_tensor(img_np):
    img_np = (img_np - MEAN) / STD
    return torch.from_numpy(img_np.transpose(2, 0, 1)).float()


def build_batch(base_np, fill_np, order, step_sizes, h, w, n_steps):
    total_pixels = h * w
    mask_flat = np.zeros(total_pixels, dtype=bool)
    batch = []

    for i in range(n_steps + 1):
        if i > 0:
            idxs = order[step_sizes[i - 1]:step_sizes[i]]
            mask_flat[idxs] = True
        mask_2d = mask_flat.reshape(h, w)
        composed = np.where(mask_2d[..., None], fill_np, base_np)
        batch.append(normalize_tensor(composed.astype(np.float32)))

    return torch.stack(batch)


def compute_insertion_deletion(orig_img_pil, cam, target_class_idx,
                                prototypes_norm, n_steps=N_STEPS):

    img_np = np.array(orig_img_pil).astype(np.float32) / 255.0
    h, w = img_np.shape[:2]

    blur_np  = cv2.GaussianBlur(img_np, (BLUR_KSIZE, BLUR_KSIZE), 0)
    black_np = np.zeros_like(img_np)

    cam_flat = cam.flatten()
    order = np.argsort(-cam_flat)              # most important pixel first
    total_pixels = len(order)
    step_sizes = np.linspace(0, total_pixels, n_steps + 1).astype(int)

    # INSERTION: starts blurred -> reveals original pixels (most important first)
    insertion_batch = build_batch(blur_np, img_np, order, step_sizes, h, w, n_steps)

    # DELETION: starts original -> blacks out pixels (most important first)
    deletion_batch = build_batch(img_np, black_np, order, step_sizes, h, w, n_steps)

    with torch.no_grad():
        feat_ins = ssl_model.encoder(insertion_batch.to(device))
        feat_ins = F.normalize(feat_ins, dim=1)
        dist_ins = torch.cdist(feat_ins, prototypes_norm)
        probs_ins = torch.softmax(-dist_ins, dim=1)[:, target_class_idx].cpu().numpy()

        feat_del = ssl_model.encoder(deletion_batch.to(device))
        feat_del = F.normalize(feat_del, dim=1)
        dist_del = torch.cdist(feat_del, prototypes_norm)
        probs_del = torch.softmax(-dist_del, dim=1)[:, target_class_idx].cpu().numpy()

    x = np.linspace(0, 1, n_steps + 1)
    insertion_auc = np.trapz(probs_ins, x)
    deletion_auc  = np.trapz(probs_del, x)

    return probs_ins, probs_del, insertion_auc, deletion_auc, x


# -----------------------------------------------------
# RUN OVER SAMPLE IMAGES (one correctly-classified image per class)
# reuses `sample_indices`, `prototypes_norm`, `class_names` from CELL-5
# -----------------------------------------------------

results = []
curves_gc  = {"insertion": [], "deletion": []}
curves_gcpp = {"insertion": [], "deletion": []}

for cls_idx, idx in sorted(sample_indices.items()):

    img_path = test_ds.images[idx]
    orig_img = Image.open(img_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))

    # ---------- Grad-CAM ----------
    img_tensor = test_transform(orig_img).unsqueeze(0).to(device)
    img_tensor.requires_grad_(True)
    ssl_model.zero_grad()
    cam_extractor = GradCAM(ssl_model, target_layer)
    score = protonet_score(img_tensor, cls_idx, prototypes_norm)
    cam_map = cam_extractor.generate(score, img_size=(IMG_SIZE, IMG_SIZE))
    cam_extractor.remove_hooks()

    ins_gc, del_gc, ins_auc_gc, del_auc_gc, x = compute_insertion_deletion(
        orig_img, cam_map, cls_idx, prototypes_norm
    )

    # ---------- Grad-CAM++ ----------
    img_tensor2 = test_transform(orig_img).unsqueeze(0).to(device)
    img_tensor2.requires_grad_(True)
    ssl_model.zero_grad()
    campp_extractor = GradCAMPlusPlus(ssl_model, target_layer)
    score2 = protonet_score(img_tensor2, cls_idx, prototypes_norm)
    campp_map = campp_extractor.generate(score2, img_size=(IMG_SIZE, IMG_SIZE))
    campp_extractor.remove_hooks()

    ins_gcpp, del_gcpp, ins_auc_gcpp, del_auc_gcpp, _ = compute_insertion_deletion(
        orig_img, campp_map, cls_idx, prototypes_norm
    )

    curves_gc["insertion"].append(ins_gc)
    curves_gc["deletion"].append(del_gc)
    curves_gcpp["insertion"].append(ins_gcpp)
    curves_gcpp["deletion"].append(del_gcpp)

    results.append({
        "Class": class_names[cls_idx],
        "GradCAM_Insertion_AUC":   ins_auc_gc,
        "GradCAM_Deletion_AUC":    del_auc_gc,
        "GradCAM++_Insertion_AUC": ins_auc_gcpp,
        "GradCAM++_Deletion_AUC":  del_auc_gcpp,
    })

# -----------------------------------------------------
# RESULTS TABLE
# -----------------------------------------------------

results_df = pd.DataFrame(results)

mean_row = {
    "Class": "MEAN",
    "GradCAM_Insertion_AUC":   results_df["GradCAM_Insertion_AUC"].mean(),
    "GradCAM_Deletion_AUC":    results_df["GradCAM_Deletion_AUC"].mean(),
    "GradCAM++_Insertion_AUC": results_df["GradCAM++_Insertion_AUC"].mean(),
    "GradCAM++_Deletion_AUC":  results_df["GradCAM++_Deletion_AUC"].mean(),
}
results_df = pd.concat([results_df, pd.DataFrame([mean_row])], ignore_index=True)

print("\n" + "=" * 70)
print("INSERTION / DELETION AUC  (Insertion ↑ better | Deletion ↓ better)")
print("=" * 70)
print(results_df.round(4).to_string(index=False))

metrics_csv = os.path.join(METRIC_DIR, "insertion_deletion_metrics.csv")
results_df.to_csv(metrics_csv, index=False)
print(f"\n✅ Saved: {metrics_csv}")

# -----------------------------------------------------
# AVERAGE CURVES PLOT
# -----------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

mean_ins_gc   = np.mean(curves_gc["insertion"], axis=0)
mean_del_gc   = np.mean(curves_gc["deletion"], axis=0)
mean_ins_gcpp = np.mean(curves_gcpp["insertion"], axis=0)
mean_del_gcpp = np.mean(curves_gcpp["deletion"], axis=0)

axes[0].plot(x, mean_ins_gc, label="Grad-CAM", marker="o", markersize=3)
axes[0].plot(x, mean_ins_gcpp, label="Grad-CAM++", marker="s", markersize=3)
axes[0].set_title("Insertion Curve (↑ better)")
axes[0].set_xlabel("Fraction of pixels inserted")
axes[0].set_ylabel("Prototype probability")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(x, mean_del_gc, label="Grad-CAM", marker="o", markersize=3)
axes[1].plot(x, mean_del_gcpp, label="Grad-CAM++", marker="s", markersize=3)
axes[1].set_title("Deletion Curve (↓ better)")
axes[1].set_xlabel("Fraction of pixels deleted")
axes[1].set_ylabel("Prototype probability")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()

curve_fig = os.path.join(METRIC_DIR, "insertion_deletion_curves.png")
plt.savefig(curve_fig, dpi=600, bbox_inches="tight")
plt.show()

print(f"✅ Saved: {curve_fig}")